# TEMptation validation — normal vs. pathological

**Not a diagnostic tool.** Every metric here is a morphometric measurement derived from a segmentation mask, not validated for or intended for clinical/research diagnosis. See `docs/pathology_score.md`.

**Sanity, not significance.** This notebook reproduces the checks in `docs/validation_report.md` and `tests/test_biological_sanity.py`. No hypothesis test, no p-value, no ROC curve, no classifier accuracy figure appears anywhere here, on purpose (CLAUDE.md §9/§20).

Run this notebook with the `nerve_env` conda environment (the same one the test suite uses): `conda activate nerve_env && jupyter notebook notebooks/validation.ipynb`. The outputs embedded below are from a real run against this repository's `normal_data`/`pathological_data`, captured while writing `docs/validation_report.md` — re-running should reproduce them exactly (the pipeline is deterministic).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from temptation import discovery, pipeline
from temptation.config import QCThresholds, SegmentationConfig

DATA_ROOT = Path("..").resolve().parent  # repo root, containing normal_data/ and pathological_data/
GROUP_MAP = {"normal_data": "normal", "pathological_data": "pathological"}

pairs, skipped = discovery.find_groups(DATA_ROOT, GROUP_MAP)
print(f"{len(pairs)} pairs, {len(skipped)} skipped")

result = pipeline.analyze_dataset(
    pairs, pixel_length_um=0.00524,
    seg_cfg=SegmentationConfig(), qc_thresholds=QCThresholds(),
)
print("axons:", len(result.df_axons), "images:", len(result.df_image), "errors:", len(result.errors))

df = result.df_axons
img = result.df_image
normal = df[df["group"] == "normal"]
patho = df[df["group"] == "pathological"]

100 pairs, 0 skipped
axons: 1165 images: 100 errors: 0


## 1. CLAUDE.md §13 invariants (normal group)

In [2]:
print("normal axons:", len(normal), "pathological axons:", len(patho))
print()

g = normal["g_ratio"].dropna()
print(f"g_ratio: min={g.min():.4f} max={g.max():.4f} mean={g.mean():.4f} n_valid={len(g)}/{len(normal)}")
print(f"g_ratio out of [0,1]: {((g<0)|(g>1)).sum()}")

fa, aa = normal["fiber_area_um2"], normal["axon_area_um2"]
print(f"fiber_area > axon_area violations: {(fa <= aa).sum()} / {len(normal)}")

ma = normal["myelin_area_um2"]
resid = (ma - (fa - aa)).abs()
print(f"myelin = fiber-axon max abs residual: {resid.max():.6e}")

occ = normal["mito_occupancy_ratio"].dropna()
print(f"mito_occupancy_ratio: min={occ.min():.4f} max={occ.max():.4f}  out of [0,1]: {((occ<0)|(occ>1)).sum()}")

nml = normal["normalized_mito_load"].dropna()
print(f"normalized_mito_load: min={nml.min():.4f} max={nml.max():.4f}  out of [0,1]: {((nml<0)|(nml>1)).sum()}")

irr = normal["axon_shape_irregularity"].dropna()
print(f"axon_shape_irregularity: min={irr.min():.4f}  <1: {(irr<1).sum()}")
irr_c = normal["axon_shape_irregularity_crofton"].dropna()
print(f"axon_shape_irregularity_crofton: min={irr_c.min():.4f}  <1: {(irr_c<1).sum()}")

circ = normal["circularity"].dropna()
print(f"circularity: max={circ.max():.4f}  >1: {(circ>1).sum()}")
circ_c = normal["axon_circularity_crofton"].dropna()
print(f"axon_circularity_crofton: max={circ_c.max():.4f}  >1.05: {(circ_c>1.05).sum()}")

normal axons: 1140 pathological axons: 25

g_ratio: min=0.2866 max=0.8748 mean=0.6794 n_valid=1140/1140
g_ratio out of [0,1]: 0
fiber_area > axon_area violations: 0 / 1140
myelin = fiber-axon max abs residual: 8.881784e-16
mito_occupancy_ratio: min=0.0000 max=0.8526  out of [0,1]: 0
normalized_mito_load: min=0.0000 max=0.4883  out of [0,1]: 0
axon_shape_irregularity: min=1.1036  <1: 0
axon_shape_irregularity_crofton: min=1.0161  <1: 0
circularity: max=0.9062  >1: 0
axon_circularity_crofton: max=0.9841  >1.05: 0


All invariants hold. `circularity`'s max of 0.906 (not 1.0) is expected, not a bug — see `docs/known_issues.md` F5.

## 2. Pre-registered baseline: mean g-ratio

Blueprint's original audit estimated ~0.63 from a 4-image subset (`normal_data/162-165`). Checking the full normal group (this is a **pre-registered range check**, not a threshold picked after seeing the number).

In [3]:
mean_g_ratio = normal["g_ratio"].mean()
print(f"mean_g_ratio = {mean_g_ratio:.4f}")
print("within documented range [0.55, 0.75]:", 0.55 <= mean_g_ratio <= 0.75)

mean_g_ratio = 0.6794
within documented range [0.55, 0.75]: True


## 3. Documented degeneracies on the pathological group (CLAUDE.md F4)

Myelin is fully detached in this group. `axon_vol_fraction` should collapse to exactly 1.0, and every myelin-derived column should be entirely `NaN`.

In [4]:
print("axon_vol_fraction unique values:", sorted(patho["axon_vol_fraction"].unique()))
print("myelin_area_um2 all NaN:", patho["myelin_area_um2"].isna().all(), f" (n={len(patho)})")
print("g_ratio all NaN:", patho["g_ratio"].isna().all())
print("qc_no_myelin fires on every axon:", bool(patho["qc_no_myelin"].all()))

axon_vol_fraction unique values: [1.0]
myelin_area_um2 all NaN: True  (n=25)
g_ratio all NaN: True
qc_no_myelin fires on every axon: True


Confirmed exactly as documented — and `qc_no_myelin` (permanently non-excludable, see `docs/qc.md`) catches every one of these axons, so `--exclude-qc-failed` can never silently delete this entire group.

## 4. `image_demyelination_index` group separation

Needs an externally-supplied reference (no CLI default — CLAUDE.md §11 A4). For this notebook only, using the normal group's own median `myelin_area_fraction_of_fov` as a convenience reference.

In [5]:
normal_img = img[img["group"] == "normal"]
patho_img = img[img["group"] == "pathological"]

reference = normal_img["myelin_area_fraction_of_fov"].median()
print(f"reference (median normal myelin_area_fraction_of_fov): {reference:.4f}")

def demyelination_index(frac, ref):
    if ref is None or ref == 0 or np.isnan(ref):
        return np.nan
    return float(np.clip(1 - frac / ref, 0, 1))

normal_di = normal_img["myelin_area_fraction_of_fov"].apply(lambda f: demyelination_index(f, reference))
patho_di = patho_img["myelin_area_fraction_of_fov"].apply(lambda f: demyelination_index(f, reference))

print(f"normal demyelination_index:       mean={normal_di.mean():.4f} min={normal_di.min():.4f} max={normal_di.max():.4f}")
print(f"pathological demyelination_index: mean={patho_di.mean():.4f} min={patho_di.min():.4f} max={patho_di.max():.4f}")
print("strict separation (pathological min > normal max):", patho_di.min() > normal_di.max())

reference (median normal myelin_area_fraction_of_fov): 0.2641
normal demyelination_index:       mean=0.1166 min=0.0000 max=0.7274
pathological demyelination_index: mean=0.9500 min=0.8843 max=1.0000
strict separation (pathological min > normal max): True


## 5. Pathology-score component distributions — why three stay blocked

`docs/pathology_score.md` lists three candidate score components with no calibrated threshold. Checking whether real distributions from this dataset supply enough basis to unblock any of them.

In [6]:
frag_n = normal["mito_fragmentation_index"].dropna()
frag_p = patho["mito_fragmentation_index"].dropna()
print("=== mito_fragmentation_index (axon-level) ===")
print(f"normal:       n_valid={len(frag_n)} median={frag_n.median():.2f} p90={frag_n.quantile(0.9):.2f} max={frag_n.max():.2f}")
print(f"pathological: n_valid={len(frag_p)}  median={frag_p.median():.2f}")
print()

print("=== mito_occupancy_ratio (axon-level) ===")
print(f"normal:       mean={normal['mito_occupancy_ratio'].mean():.4f} median={normal['mito_occupancy_ratio'].median():.4f} max={normal['mito_occupancy_ratio'].max():.4f}")
print(f"pathological: mean={patho['mito_occupancy_ratio'].mean():.4f} median={patho['mito_occupancy_ratio'].median():.4f} max={patho['mito_occupancy_ratio'].max():.4f}  (n={len(patho)})")
print("NOTE: pathological mean is LOWER than normal -- opposite of a naive")
print("'high occupancy = pathology' assumption. See docs/validation_report.md §5.")

=== mito_fragmentation_index (axon-level) ===
normal:       n_valid=592 median=23.15 p90=267.60 max=1655.45
pathological: n_valid=10  median=15.56

=== mito_occupancy_ratio (axon-level) ===
normal:       mean=0.0692 median=0.0083 max=0.8526
pathological: mean=0.0417 median=0.0000 max=0.2297  (n=25)
NOTE: pathological mean is LOWER than normal -- opposite of a naive
'high occupancy = pathology' assumption. See docs/validation_report.md §5.


**Conclusion**: none of the three blocked components (`low_myelin_fraction`, `high_mito_fragmentation`, `high_mito_occupancy`) gained a defensible threshold from this pass — `mito_fragmentation_index` is heavily skewed with a small valid-n subset, `mito_occupancy_ratio` shows no clear (or even the *expected*) directional signal in this dataset, and `low_myelin_fraction` hits the same structural NaN issue as §4. See `docs/validation_report.md` §6 for the full recommendation.

**Do not** tune thresholds until the two groups separate — that is a hand-fitted classifier, explicitly out of scope (CLAUDE.md §9/§20).